# Track 1 — Legacy Well Log Digitisation

Optional inspection and validation starter. **Not** an OCR, curve extraction or calibration solution. Use the approved dataset provided by the organisers. The notebook locates it when run from the repository root or this starter-kit directory; otherwise set `DATA_ROOT` to its location.

In [ ]:
from pathlib import Path
import csv
from PIL import Image
from IPython.display import display

# Resolve data from either the repository root or the starter-kit directory.
def find_data_root():
    candidates = [
        Path.cwd() / "data" / "track_1",
        Path.cwd() / "../../data/track_1",
        Path.cwd() / "../data/track_1",
    ]
    for candidate in candidates:
        if candidate.is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Cannot locate data/track_1. Run from the repository root or the track starter-kit "
        "directory, or set DATA_ROOT to the approved dataset location."
    )

DATA_ROOT = find_data_root()
METADATA_DIR = DATA_ROOT / "metadata"
TEMPLATE = METADATA_DIR / "target_depths_template.csv"
IMAGE_DIR = DATA_ROOT / "log_images"
EXPECTED_COLUMNS = ["page_id", "well", "depth_m", "curve", "value", "confidence"]
print("Dataset found:", DATA_ROOT.exists())
print("Template found:", TEMPLATE.exists())
print("Images folder found:", IMAGE_DIR.exists())

# Load the shared submission checker (submission/validate_submission.py in this repository).
import sys
import json

def find_validator_dir():
    for base in (Path.cwd(), Path.cwd() / "..", Path.cwd() / "../.."):
        folder = base / "submission"
        if (folder / "validate_submission.py").is_file():
            return folder.resolve()
    raise FileNotFoundError(
        "Cannot locate submission/validate_submission.py. Run from the repository root or the track starter-kit directory."
    )

sys.path.insert(0, str(find_validator_dir()))
import validate_submission as vs
print("Submission checker loaded:", vs.__file__)

In [ ]:
targets = []
if TEMPLATE.is_file():
    with TEMPLATE.open(newline="", encoding="utf-8-sig") as handle:
        reader = csv.DictReader(handle)
        actual_columns = reader.fieldnames or []
        missing = [c for c in EXPECTED_COLUMNS if c not in actual_columns]
        print("Template columns:", actual_columns)
        print("Missing required columns:", missing)
        if not missing:
            targets = list(reader)
            print("Target rows:", len(targets))
            print("Sample target rows:", targets[:3])
else:
    print("No template at the placeholder path. Ask technical support for the approved data root.")

In [ ]:
images = sorted(
    p for p in IMAGE_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in {".png", ".jpg", ".jpeg", ".tif", ".tiff"}
) if IMAGE_DIR.is_dir() else []
print("Raster image files:", len(images))
if images:
    try:
        with Image.open(images[0]) as image:
            print("First image:", images[0].name, image.size, image.mode)
            display(image.copy())
    except (OSError, ValueError) as exc:
        print("Image could not be previewed:", exc)
else:
    print("No raster images found at this path; supplied PDFs may require separate local processing.")

## Output-contract validator

Once **your own extraction code** has produced a CSV, set `PREDICTIONS` to its location and run this check. It checks columns, page IDs and wells, and that every target depth has a reading within 0.10 m. It does **not** check extraction accuracy. Start from `submission/template/outputs/track1_predictions_template.csv`, which lists every target, or submit your full extracted logs: only the target depths are scored, and rows between them are ignored. Leave `value` blank where you cannot extract a reading; `notes` is optional.

In [ ]:
PREDICTIONS = Path("outputs/track1_predictions.csv")  # set after implementing your solution
if PREDICTIONS.is_file():
    result = vs.validate_csv(PREDICTIONS, 1, vs.load_expected(1, DATA_ROOT))
    print(json.dumps(result, indent=2))
    print("PASS: output format valid" if result["valid"] else "FAIL: fix errors above")
else:
    print("Validator skipped: supply your own output CSV first.")

## Your implementation

Implement image/PDF handling, pixel-to-depth/value calibration, curve tracing, QC, confidence and an inference entry point. Freeze fitted/calibrated artefacts before submission; ensure your submitted notebook is reproducible. See the Technical Guide and Submission Guide.

## Validate Your Submission (optional)

Set `SUBMISSION_DIR` to your private team repository's `submission/` folder, then run the cell below. It checks required files, columns and identifiers, not answer quality or solution execution. The same check runs from a terminal at the repository root:

```bash
python submission/validate_submission.py --track 1 ../nape-aice-2026-pipeline-x/submission
```

In [ ]:
# Example only: replace pipeline-x and adjust for your current working directory.
SUBMISSION_DIR = Path("../nape-aice-2026-pipeline-x/submission")
report = vs.validate_submission(SUBMISSION_DIR, track=1, data_root=DATA_ROOT)
print(json.dumps(report, indent=2))
print("PASS: format valid" if report["valid"] else "FAIL: fix errors above")